In [1]:
import pandas as pd
import numpy as np

# Merging Datasets of NMT 2024 and selected datapoints from Derzhstat 2024/2025

## 1.  General Secondary Education in Ukraine 2024/2025 
#### Selected excel sheets:

1. Number of general secondary educational institutions at the beginning of 2024/25 academic year by region - 6
2. Number of general secondary educational institutions in urban areas at the beginning of 2024/25 academic year by region - 11
3. Number of general secondary educational institutions in rural areas at the beginning of 2024/25 academic year by region - 12
4. Number of pupils and teachers per general secondary educational institution at the beginning of 2024/25 academic year by regions - 18
5. Number of pupils in general secondary educational institutions at the beginning of 2024/25 academic year by region - 19
6. Number of pupils in general secondary educational institutions in urban areas  at the beginning of 2024/25 academic year by region - 20
7. Number of pupils in general secondary educational institutions in rural areas at the beginning of 2024/25 academic year by region - 21
8. Organization of transportation of children to school at the beginning of 2024/25 academic year by region - 56

In [2]:
FILE = 'data/derzhstat2024_secondary_education.xlsx'
DASH = '\u2212'  # U+2212 — the statistical minus sign used for missing/zero values

def clean_df(df):
    """Replace the statistical dash (−) with NaN and reset index."""
    return df.replace(DASH, np.nan).reset_index(drop=True)

def read_simple(sheet, start_row, col_names):
    """Read sheets with a single flat table (sheets 6, 18, 56)."""
    df = pd.read_excel(FILE, sheet_name=sheet, header=None)
    df = df.iloc[start_row:].dropna(subset=[0])
    df.columns = col_names
    return clean_df(df)

def read_split(sheet, start_row, left_cols, right_cols):
    """
    Read wide sheets that are physically split across 16 columns (sheets 11, 12, 19, 20, 21).
    Left half  = cols 0–7  (region_ua | data cols | region_en)
    Right half = cols 8–15 (region_ua | data cols | region_en)  ← continuation
    """
    df = pd.read_excel(FILE, sheet_name=sheet, header=None)
    df = df.iloc[start_row:].dropna(subset=[0])
    left  = df.iloc[:, :8].copy()
    right = df.iloc[:, 8:].copy()
    right.columns = range(8)
    left.columns  = left_cols
    right.columns = right_cols
    right = right.drop(columns=['region_ua_r', 'region_en_r'])
    return clean_df(pd.concat([left, right], axis=1))


# ── Sheet 6 ──────────────────────────────────────────────────────────────────
# Number of institutions (all areas), simple 11-col layout, data starts row 8
df6 = read_simple(
    sheet='6', start_row=8,
    col_names=[
        'region_ua',
        'total_institutions',
        'total_pupils', 'total_teachers',
        'public_institutions',
        'public_pupils', 'public_teachers',
        'private_institutions',
        'private_pupils', 'private_teachers',
        'region_en',
    ]
)

# ── Sheets 11 & 12 ───────────────────────────────────────────────────────────
# Institutions by type, urban / rural — wide/split 16-col layout, data starts row 8
_inst_left  = ['region_ua', 'total',
               'primary_w_preschool', 'primary',
               'gymnasium_w_preschool_primary', 'gymnasium_w_primary', 'gymnasium',
               'region_en']
_inst_right = ['region_ua_r',
               'lyceum_w_preschool_primary_gymnasium', 'lyceum_w_primary_gymnasium',
               'lyceum_w_preschool_gymnasium', 'lyceum_w_gymnasium', 'lyceum',
               'special', 'region_en_r']

df11 = read_split('11', start_row=8, left_cols=_inst_left,   right_cols=_inst_right)  # urban
df12 = read_split('12', start_row=8, left_cols=_inst_left,   right_cols=_inst_right)  # rural

# ── Sheet 18 ─────────────────────────────────────────────────────────────────
# Pupils & teachers per institution, simple 8-col layout, data starts row 6
df18 = read_simple(
    sheet='18', start_row=6,
    col_names=[
        'region_ua',
        'pupils_per_inst_all',     'teachers_per_inst_all',
        'pupils_per_inst_public',  'teachers_per_inst_public',
        'pupils_per_inst_private', 'teachers_per_inst_private',
        'region_en',
    ]
)

# ── Sheets 19, 20, 21 ────────────────────────────────────────────────────────
# Pupil counts by institution type (all / urban / rural) — same split layout, data starts row 8
_pupils_left  = ['region_ua', 'total',
                 'primary_w_preschool', 'primary',
                 'gymnasium_w_preschool_primary', 'gymnasium_w_primary', 'gymnasium',
                 'region_en']
_pupils_right = ['region_ua_r',
                 'lyceum_w_preschool_primary_gymnasium', 'lyceum_w_primary_gymnasium',
                 'lyceum_w_preschool_gymnasium', 'lyceum_w_gymnasium', 'lyceum',
                 'special', 'region_en_r']

df19 = read_split('19', start_row=8, left_cols=_pupils_left, right_cols=_pupils_right)  # all
df20 = read_split('20', start_row=8, left_cols=_pupils_left, right_cols=_pupils_right)  # urban
df21 = read_split('21', start_row=8, left_cols=_pupils_left, right_cols=_pupils_right)  # rural

# ── Sheet 56 ─────────────────────────────────────────────────────────────────
# Transportation, simple 8-col layout, data starts row 6
df56 = read_simple(
    sheet='56', start_row=6,
    col_names=[
        'region_ua',
        'urban_rural_need_transport', 'urban_rural_transport_organized', 'urban_rural_school_bus',
        'rural_need_transport',       'rural_transport_organized',       'rural_school_bus',
        'region_en',
    ]
)

region_map = {
    'Вінницька': 'Вінницька область',
    'Волинська': 'Волинська область',
    'Дніпропетровська': 'Дніпропетровська область',
    'Донецька': 'Донецька область',
    'Житомирська': 'Житомирська область',
    'Закарпатська': 'Закарпатська область',
    'Запорізька': 'Запорізька область',
    'Івано-Франківська': 'Івано-Франківська область',
    'Київська': 'Київська область',
    'Кіровоградська': 'Кіровоградська область',
    'Луганська': 'Луганська область',
    'Львівська': 'Львівська область',
    'Миколаївська': 'Миколаївська область',
    'Одеська': 'Одеська область',
    'Полтавська': 'Полтавська область',
    'Рівненська': 'Рівненська область',
    'Сумська': 'Сумська область',
    'Тернопільська': 'Тернопільська область',
    'Харківська': 'Харківська область',
    'Херсонська': 'Херсонська область',
    'Хмельницька': 'Хмельницька область',
    'Черкаська': 'Черкаська область',
    'Чернівецька': 'Чернівецька область',
    'Чернігівська': 'Чернігівська область',
    'м. Київ': 'м.Київ',   # залишаємо як є
    'Україна': 'Україна'
}
df6['region_ua_fixed'] = df6['region_ua'].map(region_map)
df56
df56['region_ua_fixed'] = df56['region_ua'].map(region_map)
df11['region_ua_fixed'] = df11['region_ua'].map(region_map)
df12['region_ua_fixed'] = df12['region_ua'].map(region_map)
df18['region_ua_fixed'] = df18['region_ua'].map(region_map)
df19['region_ua_fixed'] = df19['region_ua'].map(region_map)
df20['region_ua_fixed'] = df20['region_ua'].map(region_map)
df21['region_ua_fixed'] = df21['region_ua'].map(region_map)

df11 = df11.add_suffix('_urban')
df12 = df12.add_suffix('_rural')
df19 = df19.add_suffix('_all')
df20 = df20.add_suffix('_urban')
df21 = df21.add_suffix('_rural')
df11 = df11.rename(columns={'region_ua_fixed_urban': 'region_ua_fixed'})
df12 = df12.rename(columns={'region_ua_fixed_rural': 'region_ua_fixed'})
df19 = df19.rename(columns={'region_ua_fixed_all': 'region_ua_fixed'})
df20 = df20.rename(columns={'region_ua_fixed_urban': 'region_ua_fixed'})
df21 = df21.rename(columns={'region_ua_fixed_rural': 'region_ua_fixed'})

In [3]:
df56

,region_ua,urban_rural_need_transport,urban_rural_transport_organized,urban_rural_school_bus,rural_need_transport,rural_transport_organized,rural_school_bus,region_en,region_ua_fixed
0,Україна,397680,358232,312346,260339,240601,217988,Ukraine,Україна
1,Вінницька,24341,22513,17815,14264,13526,11199,Vinnytsya,Вінницька область
2,Волинська,19977,16831,15059,14044,13550,12322,Volyn,Волинська область
3,Дніпропетровська,25432,21154,19709,15320,13505,13341,Dnipropetrovsk,Дніпропетровська область
4,Донецька,4,NaN,NaN,4,NaN,NaN,Donetsk,Донецька область
5,Житомирська,21062,20410,17626,13772,13482,11590,Zhytomyr,Житомирська область
6,Закарпатська,24223,18643,16078,17779,14205,12575,Zakarpattya,Закарпатська область
7,Запорізька,2709,2034,2021,1807,1369,1356,Zaporizhzhya,Запорізька область
8,Івано-Франківська,21519,16891,15299,15041,11751,10650,Ivano-Frankivsk,Івано-Франківська область
9,Київська,37265,37200,27832,23685,23620,20027,Kyiv,Київська область


In [4]:
df6

,region_ua,total_institutions,total_pupils,total_teachers,public_institutions,public_pupils,public_teachers,private_institutions,private_pupils,private_teachers,region_en,region_ua_fixed
0,Україна,12291,3743887,380877,11781,3646129,369850,510,97758,11027,Ukraine,Україна
1,Вінницька,602,158670,17984,590,157120,17641,12,1550,343,Vinnytsya,Вінницька область
2,Волинська,518,137331,16611,513,136776,16509,5,555,102,Volyn,Волинська область
3,Дніпропетровська,820,308580,25788,797,306467,25439,23,2113,349,Dnipropetrovsk,Дніпропетровська область
4,Донецька,256,76768,7051,255,76694,7040,1,74,11,Donetsk,Донецька область
5,Житомирська,511,129897,14765,506,129084,14639,5,813,126,Zhytomyr,Житомирська область
6,Закарпатська,565,159143,17649,548,156929,17287,17,2214,362,Zakarpattya,Закарпатська область
7,Запорізька,318,114930,9883,313,114044,9767,5,886,116,Zaporizhzhya,Запорізька область
8,Івано-Франківська,550,154733,19607,540,152992,19323,10,1741,284,Ivano-Frankivsk,Івано-Франківська область
9,Київська,630,235647,20977,554,227211,19672,76,8436,1305,Kyiv,Київська область


In [5]:
import chardet

with open('data/Odata2024File.csv', 'rb') as f:
    result = chardet.detect(f.read(100_000))  # sample first 100k bytes
    print(result)  # e.g. {'encoding': 'UTF-8-SIG', 'confidence': 0.99}

{'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': 'uk', 'mime_type': 'text/plain'}


In [ ]:
df_nmt = pd.read_csv(
    "data/Odata2024File.csv",
    sep=";",
    quotechar='"',
    engine="python",
    on_bad_lines="skip",
    encoding="utf-8"
)


In [6]:
nmt_data = pd.read_csv(
    'data/Odata2024File.csv',
    sep=';',
    encoding='utf-8-sig',
    engine="python",  
    quotechar='"',
    na_values=['null'],
    on_bad_lines='skip'

)

nmt_data.columns = nmt_data.columns.str.strip('"')

print(df6['region_ua_fixed'].unique())
print(nmt_data['RegName'].unique())



<StringArray>
[                  'Україна',         'Вінницька область',
         'Волинська область',  'Дніпропетровська область',
          'Донецька область',       'Житомирська область',
      'Закарпатська область',        'Запорізька область',
 'Івано-Франківська область',          'Київська область',
    'Кіровоградська область',         'Луганська область',
         'Львівська область',      'Миколаївська область',
           'Одеська область',        'Полтавська область',
        'Рівненська область',           'Сумська область',
     'Тернопільська область',        'Харківська область',
        'Херсонська область',       'Хмельницька область',
         'Черкаська область',       'Чернівецька область',
      'Чернігівська область',                    'м.Київ']
Length: 26, dtype: str
<StringArray>
[        'Черкаська область',  'Дніпропетровська область',
 'Івано-Франківська область',         'Львівська область',
      'Закарпатська область',        'Запорізька область',
     

In [7]:
def drop_region_cols(df):
    return df.drop(
        columns=[col for col in df.columns if col.startswith('region_')],
        errors='ignore'
    )

df_final_1 = nmt_data.merge(
    df6,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left')
df_final_1 = drop_region_cols(df_final_1)

df_final_2 = df_final_1.merge(
    df56,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left')
df_final_2 = drop_region_cols(df_final_2)

df_final_3 = df_final_2.merge(
    df11,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left'
)
df_final_3 = drop_region_cols(df_final_3)

df_final_4 = df_final_3.merge(
    df12,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left'
)
df_final_4 = drop_region_cols(df_final_4)

df_final_5 = df_final_4.merge(
    df18,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left'
)
df_final_5 = drop_region_cols(df_final_5)

df_final_6 = df_final_5.merge(
    df19,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left'
)
df_final_6 = drop_region_cols(df_final_6)

df_final_7 = df_final_6.merge(
    df20,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left'
)
df_final_7 = drop_region_cols(df_final_7)

df_final_8 = df_final_7.merge(
    df21,
    left_on='RegName',
    right_on='region_ua_fixed',
    how='left'
)
df_final_8 = drop_region_cols(df_final_8)

df_final_8.columns[:100]


Index(['outid', 'Birth', 'SexTypeName', 'RegName', 'AreaName', 'TerName',
       'RegTypeName', 'TerTypeName', 'EOName', 'EOTypeName', 'EORegName',
       'EOAreaName', 'EOTerName', 'EOParent', 'Test', 'TestDate', 'UkrBlock',
       'UkrBlockStatus', 'UkrBlockBall100', 'UkrBlockBall', 'HistBlock',
       'HistBlockLang', 'HistBlockStatus', 'HistBlockBall100', 'HistBlockBall',
       'MathBlock', 'MathBlockLang', 'MathBlockStatus', 'MathBlockBall100',
       'MathBlockBall', 'PhysBlock', 'PhysBlockLang', 'PhysBlockStatus',
       'PhysBlockBall100', 'PhysBlockBall', 'ChemBlock', 'ChemBlockLang',
       'ChemBlockStatus', 'ChemBlockBall100', 'ChemBlockBall', 'BioBlock',
       'BioBlockLang', 'BioBlockStatus', 'BioBlockBall100', 'BioBlockBall',
       'GeoBlock', 'GeoBlockLang', 'GeoBlockStatus', 'GeoBlockBall100',
       'GeoBlockBall', 'EngBlock', 'EngBlockStatus', 'EngBlockBall100',
       'EngBlockBall', 'FraBlock', 'FraBlockStatus', 'FraBlockBall100',
       'FraBlockBall', 'DeuBloc

In [8]:
df_final_8['average_school_size'] = (
    df_final_8['total_pupils'] / df_final_8['total_institutions'])
df_final_8['transport_share'] = (
    df_final_8['urban_rural_transport_organized'] / df_final_8['total_pupils'])
df_final_8 = df_final_8.replace("х", np.nan)
df_final_8[:30]

,outid,Birth,SexTypeName,RegName,AreaName,TerName,RegTypeName,TerTypeName,EOName,EOTypeName,...,gymnasium_w_primary_rural_y,gymnasium_rural_y,lyceum_w_preschool_primary_gymnasium_rural_y,lyceum_w_primary_gymnasium_rural_y,lyceum_w_preschool_gymnasium_rural_y,lyceum_w_gymnasium_rural_y,lyceum_rural_y,special_rural_y,average_school_size,transport_share
0,9b995d13-de3d-47c2-9b4e-004025346a49,2006,жіноча,Черкаська область,Черкаський район,с.Ліпляве,Випускник загальноосвітнього навчального закла...,"селище, село","Комунальний заклад ""Ліплявський ліцей"" Ліплявс...",ліцей,...,2149,NaN,9273,20443,NaN,NaN,634,151,261.645833,0.133424
1,9c0a5f77-73e4-475d-aff9-006da514737b,2007,чоловіча,Дніпропетровська область,Нікопольський район,м.Нікополь,Випускник загальноосвітнього навчального закла...,місто,"Відокремлений структурний підрозділ ""Нікопольс...",заклад фахової передвищої освіти,...,2847,3136,3979,10558,NaN,539,16562,245,376.317073,0.068553
2,9bb9c262-2188-475c-949a-000b1a194cbb,2006,жіноча,Івано-Франківська область,Івано-Франківський район,м.Івано-Франківськ,Випускник загальноосвітнього навчального закла...,місто,"Фаховий коледж Закладу вищої освіти ""Університ...",заклад фахової передвищої освіти,...,12033,NaN,12595,39354,203,353,734,135,281.332727,0.109162
3,9b982071-10d6-4caa-b80a-004675647a31,2007,жіноча,Дніпропетровська область,м.Дніпро,Чечелівський район міста,Випускник загальноосвітнього навчального закла...,місто,"Комунальний заклад освіти ""Середня загальноосв...",середня загальноосвітня школа,...,2847,3136,3979,10558,NaN,539,16562,245,376.317073,0.068553
4,9b9fc04c-570f-4cb9-903c-0007200fe870,2007,жіноча,Львівська область,Самбірський район,с.Вовче,Випускник загальноосвітнього навчального закла...,"селище, село",ВОВЧЕНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬОЇ ОСВІТИ ...,середня загальноосвітня школа,...,17936,897,20430,35557,NaN,NaN,558,357,261.927577,0.127347
5,9b9dbac9-b738-43d0-ba52-003bc205caf6,2006,жіноча,Закарпатська область,Ужгородський район,с.Люта,Випускник загальноосвітнього навчального закла...,"селище, село",Лютянський заклад загальної середньої освіти I...,середня загальноосвітня школа,...,17314,321,4786,58413,NaN,1887,786,41,281.669027,0.117146
6,9b91bf64-b13e-40f3-9a3b-002816e7a2f9,2007,чоловіча,Запорізька область,м.Запоріжжя,Олександрівський район міста,Випускник загальноосвітнього навчального закла...,місто,Відокремлений структурний підрозділ «Економіко...,заклад фахової передвищої освіти,...,2068,388,1989,4153,NaN,465,929,245,361.415094,0.017698
7,9b9a9664-68e3-4a0e-9d65-0043e9828286,2007,жіноча,Тернопільська область,Тернопільський район,м.Тернопіль,Випускник загальноосвітнього навчального закла...,місто,"Тернопільський академічний ліцей ""Українська г...",ліцей,...,12121,143,6259,18836,113,NaN,171,96,184.538879,0.135434
8,9bc86790-f34b-4cb4-8917-0012b81dc69c,2003,жіноча,м.Київ,м.Київ,Шевченківський район міста,Випускник минулих років,місто,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,558.265,0.001126
9,9b92037b-d8dd-4277-a86d-00505d0e1b56,2007,жіноча,Харківська область,м.Харків,Індустріальний район міста,Випускник загальноосвітнього навчального закла...,місто,комунальний заклад «Харківський ліцей № 26 Хар...,ліцей,...,1942,NaN,8489,14136,96,NaN,NaN,87,338.735294,0.008987


In [43]:
df_final_8.to_csv("data/merged_dataset_logit.csv", index=False)